# 🧠 Notebook Complementar — TensorFlow e Keras para PLN
### Aula 2 — Tecnologias e Ferramentas

---

## 🎯 O que você vai aprender

- Compreender a relação entre **TensorFlow e Keras**
- Como representar texto para redes neurais (**Bag-of-Words, TF-IDF, Embeddings**)
- Construir e treinar um **classificador de texto** com Keras
- Usar a camada **Embedding** para aprender representações semânticas
- Entender e interpretar **métricas e curvas de aprendizado**
- Salvar e reutilizar modelos treinados

---

## 📚 Teoria: TensorFlow e Keras

### O que é o TensorFlow?

O **TensorFlow** é um framework de computação numérica e machine learning desenvolvido  
pelo **Google Brain** e lançado como open-source em novembro de 2015.  
Ele é a base sobre a qual redes neurais profundas são construídas, treinadas e implantadas.

```
TensorFlow (motor de cálculo)
     │
     └── Keras (API de alto nível — mais simples e intuitiva)
              │
              ├── Sequential API  → camadas em sequência (nosso foco)
              ├── Functional API  → grafos complexos
              └── Subclassing API → máxima flexibilidade
```

### Por que TensorFlow para PLN?

| Vantagem | Descrição |
|---|---|
| **Flexibilidade** | Cria qualquer arquitetura (RNN, LSTM, Transformer) |
| **Produção** | TF Serving, TF Lite para deploy em servidores e mobile |
| **Ecossistema** | TF Hub, TensorBoard, TF Data para pipelines completos |
| **GPU/TPU** | Aceleração automática em hardware especializado |
| **Keras** | API simples para criar modelos em poucas linhas |

### O Ciclo de Vida de um Modelo Keras

```
1. CONSTRUIR    → definir camadas e arquitetura
2. COMPILAR     → escolher optimizer, loss function e métricas
3. TREINAR      → ajustar pesos com model.fit()
4. AVALIAR      → medir performance no conjunto de teste
5. PREVER       → model.predict() em novos dados
6. SALVAR       → model.save() para reutilizar depois
```

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONFIGURAÇÃO INICIAL
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ Keras      : {keras.__version__}')
print(f'✅ NumPy      : {np.__version__}')
print()
gpus = tf.config.list_physical_devices('GPU')
print(f'🖥️  GPU disponível: {"Sim — " + str(gpus) if gpus else "Não (usando CPU)"}')
print()
print('💡 Sem GPU, todos os exemplos ainda funcionam — apenas mais lento.')

---
## 🔵 Parte 1 — Como Representar Texto para Redes Neurais

### O problema central: redes neurais só entendem números!

```
Entrada  : "O gato sentou no tapete"
           ↓
Precisamos: [0.32, -0.18, 0.74, 0.09, -0.55, ...]  (números!)
```

Existem três abordagens principais:

| Abordagem | Vantagem | Desvantagem |
|---|---|---|
| **Bag-of-Words** | Simples, rápido | Perde ordem das palavras |
| **TF-IDF** | Pondera raridade | Ainda perde ordem |
| **Embeddings** | Captura semântica e ordem | Mais complexo |

### 1.1 Bag-of-Words (BoW)

In [ ]:
# ─────────────────────────────────────────────────────────────
# BAG-OF-WORDS — Representação por Contagem
# Cada documento vira um vetor de contagens de palavras
# A ordem das palavras é IGNORADA
#
# TextVectorization(output_mode='count') → conta ocorrências
# ─────────────────────────────────────────────────────────────

corpus_bow = [
    'o gato perseguiu o rato',
    'o cachorro latiu para o gato',
    'o rato fugiu do gato e do cachorro',
    'o gato dorme no sol',
    'o cachorro corre no parque',
]

# Criar e adaptar o vetorizador
vetorizador_bow = keras.layers.TextVectorization(
    max_tokens=30,         # vocabulário máximo
    output_mode='count',   # contar ocorrências
    standardize='lower',   # converter para minúsculas
)
vetorizador_bow.adapt(corpus_bow)  # aprende o vocabulário

vocabulario = vetorizador_bow.get_vocabulary()
print(f'📚 Vocabulário aprendido ({len(vocabulario)} termos):')
print(f'   {vocabulario}')
print()

# Vetorizar cada documento
vetores = vetorizador_bow(corpus_bow).numpy()

print('📊 Matriz Bag-of-Words (documentos × palavras):')
print(f'   Shape: {vetores.shape}  ({len(corpus_bow)} documentos × {len(vocabulario)} palavras)')
print()

# Mostrar de forma legível
print(f'{"Documento":<40} ', end='')
for v in vocabulario[:8]:
    print(f'{v:<8}', end='')
print('...')
print('-' * 105)
for doc, vetor in zip(corpus_bow, vetores):
    print(f'{doc[:38]:<40} ', end='')
    for val in vetor[:8]:
        print(f'{int(val):<8}', end='')
    print('...')

# Visualizar a matriz como heatmap
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(vetores[:, 1:12], aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(11))
ax.set_xticklabels(vocabulario[1:12], rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(corpus_bow)))
ax.set_yticklabels([d[:30] for d in corpus_bow], fontsize=9)
plt.colorbar(im, ax=ax, label='Contagem')
ax.set_title('Matriz Bag-of-Words — Contagem de palavras por documento',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
### 1.2 TF-IDF — Term Frequency × Inverse Document Frequency

In [ ]:
# ─────────────────────────────────────────────────────────────
# TF-IDF — Ponderação por Raridade
#
# TF  = frequência do termo no documento
# IDF = quão raro é o termo em todos os documentos
#
# Palavras que aparecem em TODOS os docs têm IDF baixo
# Palavras RARAS têm IDF alto → mais informativas!
#
# TextVectorization(output_mode='tf-idf') → aplica automaticamente
# ─────────────────────────────────────────────────────────────

vetorizador_tfidf = keras.layers.TextVectorization(
    max_tokens=30,
    output_mode='tf_idf',
    standardize='lower',
)
vetorizador_tfidf.adapt(corpus_bow)

vetores_tfidf = vetorizador_tfidf(corpus_bow).numpy()
vocabulario_tfidf = vetorizador_tfidf.get_vocabulary()

print('📊 Comparação: Bag-of-Words vs TF-IDF')
print(f'   (usando o documento: "{corpus_bow[0]}")\n')

doc_idx = 0
print(f'{"Palavra":<15} {"BoW (contagem)":<18} {"TF-IDF (ponderado)":<20} {"Mais informativo?"}')
print('-' * 72)

for i, palavra in enumerate(vocabulario_tfidf[1:12], start=1):
    bow_val   = int(vetores[doc_idx, i])
    tfidf_val = round(float(vetores_tfidf[doc_idx, i]), 4)
    destaque  = '⭐ sim' if tfidf_val > 0.3 else ''
    print(f'{palavra:<15} {bow_val:<18} {tfidf_val:<20} {destaque}')

print()
print('💡 Observação:')
print('   Palavras como "o" aparecem em todos os documentos → IDF baixo → TF-IDF próximo de 0')
print('   Palavras raras ("perseguiu", "parque") → IDF alto → TF-IDF maior')

# Visualizar TF-IDF
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

for ax, vets, titulo, cmap in [
    (ax1, vetores[:, 1:11],       'Bag-of-Words (contagem)', 'Blues'),
    (ax2, vetores_tfidf[:, 1:11], 'TF-IDF (ponderado)',      'Oranges'),
]:
    im = ax.imshow(vets, aspect='auto', cmap=cmap, interpolation='nearest')
    ax.set_xticks(range(10))
    ax.set_xticklabels(vocabulario[1:11], rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(corpus_bow)))
    ax.set_yticklabels([d[:25] for d in corpus_bow], fontsize=9)
    ax.set_title(titulo, fontweight='bold')
    plt.colorbar(im, ax=ax)

plt.suptitle('BoW vs TF-IDF — Representação de Documentos', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
### 1.3 Sequências de IDs — Entrada para Embeddings

In [ ]:
# ─────────────────────────────────────────────────────────────
# SEQUÊNCIAS DE IDs — Preserva a ordem das palavras
# Cada palavra é mapeada para um ID inteiro
# A ORDEM é mantida → essencial para RNNs e Transformers
#
# output_sequence_length → padeia ou trunca para comprimento fixo
# 0 = padding (palavra vazia para completar o tamanho)
# ─────────────────────────────────────────────────────────────

MAX_LEN = 8  # comprimento máximo de sequência

vetorizador_seq = keras.layers.TextVectorization(
    max_tokens=30,
    output_mode='int',              # IDs inteiros
    output_sequence_length=MAX_LEN, # comprimento fixo
    standardize='lower',
)
vetorizador_seq.adapt(corpus_bow)
vocabulario_seq = vetorizador_seq.get_vocabulary()

sequencias = vetorizador_seq(corpus_bow).numpy()

print('🔢 Representação como Sequências de IDs:\n')
print(f'  {"Documento":<42} {"Sequência de IDs (comprimento="+str(MAX_LEN)+")"}')
print('  ' + '-' * 80)
for doc, seq in zip(corpus_bow, sequencias):
    print(f'  {doc:<42} {seq}')

print()
print('💡 Decodificação (ID → palavra):')
for i, palavra in enumerate(vocabulario_seq[:12]):
    print(f'   ID {i:2d} → "{palavra}"', end='  |  ' if (i+1)%3 else '\n')

print()
print('⚠️  ID 0 = "" (padding) — usado para preencher sequências curtas')
print('   ID 1 = "[UNK]" (desconhecido) — palavras fora do vocabulário')

# Visualizar sequências
fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(sequencias, aspect='auto', cmap='RdPu', interpolation='nearest')
for i in range(sequencias.shape[0]):
    for j in range(sequencias.shape[1]):
        val = sequencias[i, j]
        palavra = vocabulario_seq[val] if val < len(vocabulario_seq) else ''
        ax.text(j, i, f'{val}\n{palavra}', ha='center', va='center',
                fontsize=8, fontweight='bold' if val > 0 else 'normal',
                color='white' if val > 8 else 'black')
ax.set_title(f'Sequências de IDs — cada célula = (ID, palavra) | comprimento={MAX_LEN}',
             fontweight='bold')
ax.set_ylabel('Documentos')
ax.set_xlabel('Posição na sequência')
plt.tight_layout()
plt.show()

---
## 🟢 Parte 2 — A Camada Embedding em Detalhe

### O que a camada Embedding aprende?

A camada `Embedding` é uma **tabela de busca** onde cada palavra tem um vetor.  
Durante o treinamento, esses vetores são ajustados para que palavras  
com significados próximos fiquem próximas no espaço vetorial.

```
Vocabulário de 1000 palavras, Embedding de dimensão 16:

  gato    → [0.32, -0.18,  0.74, 0.09, -0.55, ...] (16 valores)
  cachorro→ [0.29, -0.21,  0.71, 0.12, -0.51, ...] (similar ao gato!)
  carro   → [-0.82, 0.44, -0.12, 0.73,  0.38, ...] (muito diferente)
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# CAMADA EMBEDDING — Como funciona internamente
#
# Embedding(vocab_size, embed_dim)
#   vocab_size → quantas palavras no vocabulário
#   embed_dim  → dimensão dos vetores de embedding
#
# Entrada : batch de sequências de IDs  → shape (batch, seq_len)
# Saída   : batch de matrizes de vetores → shape (batch, seq_len, embed_dim)
# ─────────────────────────────────────────────────────────────

VOCAB_SIZE = len(vocabulario_seq)
EMBED_DIM  = 8   # pequeno para visualização
SEQ_LEN    = MAX_LEN

# Criar apenas a camada de embedding
embedding_layer = keras.layers.Embedding(
    input_dim=VOCAB_SIZE,   # tamanho do vocabulário
    output_dim=EMBED_DIM,   # dimensão dos vetores
    name='embedding'
)

# Passar uma sequência pela camada
seq_exemplo = sequencias[0:1]  # primeira frase, formato (1, 8)
embeddings_saida = embedding_layer(seq_exemplo)

print(f'📄 Frase: "{corpus_bow[0]}"')
print(f'   IDs  : {seq_exemplo[0]}')
print(f'\n📐 Shape de entrada : {seq_exemplo.shape}  (1 frase, {SEQ_LEN} tokens)')
print(f'   Shape de saída  : {embeddings_saida.shape}  (1 frase, {SEQ_LEN} tokens, {EMBED_DIM} dims)')
print()
print('🔢 Matriz de Embeddings para cada token:')
print(f'  {"Posição":<8} {"ID":<6} {"Palavra":<12} {"Vetor (8 dimensões)"}')
print('  ' + '-' * 65)
for pos in range(SEQ_LEN):
    id_tok = int(seq_exemplo[0, pos])
    palavra = vocabulario_seq[id_tok] if id_tok < len(vocabulario_seq) else ''
    vetor   = embeddings_saida[0, pos, :].numpy().round(3)
    print(f'  {pos:<8} {id_tok:<6} {palavra:<12} {vetor}')

# Visualizar a matriz de embeddings
fig, ax = plt.subplots(figsize=(10, 5))
matriz = embeddings_saida[0].numpy()
im = ax.imshow(matriz.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')

palavras_seq = [vocabulario_seq[int(i)] for i in seq_exemplo[0]]
ax.set_xticks(range(SEQ_LEN))
ax.set_xticklabels(palavras_seq, fontsize=10)
ax.set_yticks(range(EMBED_DIM))
ax.set_yticklabels([f'dim {i}' for i in range(EMBED_DIM)], fontsize=9)
ax.set_xlabel('Tokens da frase (posição na sequência)', fontsize=10)
ax.set_ylabel('Dimensões do Embedding', fontsize=10)
ax.set_title('Matriz de Embeddings — cada coluna é o vetor de um token\n'
             '(valores aleatórios antes do treinamento — serão ajustados!)',
             fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---
## 🟡 Parte 3 — Modelo Completo de Classificação de Texto

### Dataset: Análise de Sentimento de Reviews de Filmes

Vamos construir um modelo para classificar reviews como **positivos** ou **negativos**.

In [ ]:
# ─────────────────────────────────────────────────────────────
# DATASET DE SENTIMENTO
# Conjunto de reviews de filmes em inglês
# Rótulo: 1 = positivo, 0 = negativo
# ─────────────────────────────────────────────────────────────

# Dataset de treino
reviews_treino = [
    # Positivos (label=1)
    'this movie is absolutely wonderful and amazing',
    'brilliant performance by all the actors loved it',
    'the plot was engaging and the ending was perfect',
    'outstanding cinematography and great storytelling',
    'one of the best films i have ever seen fantastic',
    'incredible acting and beautiful music throughout',
    'masterpiece of modern cinema highly recommended',
    'superb direction and excellent writing loved every moment',
    'wonderful story with amazing characters and great acting',
    'absolutely loved this film brilliant and touching',
    # Negativos (label=0)
    'terrible movie waste of time and money awful',
    'boring and predictable plot with bad acting poor',
    'worst film i have ever seen completely dreadful',
    'horrible story and terrible direction very disappointing',
    'this movie was a complete disaster awful and boring',
    'dreadful script and poor performances very bad',
    'painfully slow and terribly written waste of time',
    'horrible cinematography and terrible music awful film',
    'the worst movie ever made complete disaster avoided',
    'absolutely terrible and boring film very disappointing',
]
labels_treino = [1]*10 + [0]*10

# Dataset de validação
reviews_val = [
    'a wonderful and touching story loved the characters',
    'great acting and beautiful visuals highly recommend',
    'terrible plot and boring characters awful movie',
    'very disappointing film with poor acting horrible',
    'the film was decent but nothing special average',
]
labels_val = [1, 1, 0, 0, 1]

# Converter para arrays NumPy
X_treino = np.array(reviews_treino)
y_treino = np.array(labels_treino)
X_val    = np.array(reviews_val)
y_val    = np.array(labels_val)

print('📊 Dataset de Sentimento:')
print(f'   Treino    : {len(X_treino)} reviews ({sum(y_treino)} positivos, {len(y_treino)-sum(y_treino)} negativos)')
print(f'   Validação : {len(X_val)} reviews ({sum(y_val)} positivos, {len(y_val)-sum(y_val)} negativos)')
print()
print('📝 Exemplos de reviews:')
for i in [0, 10]:
    sent = 'POSITIVO 😊' if labels_treino[i] == 1 else 'NEGATIVO 😞'
    print(f'   [{sent}] "{reviews_treino[i]}"')

---
### 3.1 Construindo o Modelo

In [ ]:
# ─────────────────────────────────────────────────────────────
# CONSTRUÇÃO DO MODELO
#
# Arquitetura:
#   TextVectorization → tokeniza e converte para IDs
#   Embedding         → IDs → vetores densos (aprende semântica)
#   GlobalAvgPooling  → agrega vetores em representação única
#   Dense(16, relu)   → camada oculta (aprende padrões)
#   Dropout(0.3)      → regularização (evita overfitting)
#   Dense(1, sigmoid) → saída: probabilidade de ser positivo
# ─────────────────────────────────────────────────────────────

# Hiperparâmetros do modelo
VOCAB_SIZE = 500
EMBED_DIM  = 16
MAX_LEN    = 15
UNITS_OCULTAS = 16

# Configurar o vetorizador de texto
vetorizador = keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    standardize='lower_and_strip_punctuation',
)
vetorizador.adapt(X_treino)  # aprende vocabulário no treino

# Construir o modelo sequencial
modelo = keras.Sequential([
    # Camada 1: converter texto em IDs
    vetorizador,

    # Camada 2: aprender vetores de palavras (embeddings)
    keras.layers.Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBED_DIM,
        name='embedding'
    ),

    # Camada 3: reduzir sequência a um único vetor (média)
    keras.layers.GlobalAveragePooling1D(name='pooling'),

    # Camada 4: camada densa com ativação ReLU
    keras.layers.Dense(UNITS_OCULTAS, activation='relu', name='densa_oculta'),

    # Camada 5: Dropout — desativa 30% dos neurônios aleatoriamente no treino
    keras.layers.Dropout(0.3, name='dropout'),

    # Camada 6: saída com sigmoid → valor entre 0 e 1
    keras.layers.Dense(1, activation='sigmoid', name='saida'),
], name='classificador_sentimento')

# Compilar: definir como o modelo vai aprender
modelo.compile(
    optimizer='adam',              # algoritmo de otimização
    loss='binary_crossentropy',    # função de perda para classificação binária
    metrics=['accuracy']           # métrica de acompanhamento
)

# Mostrar arquitetura
print('🏗️ Arquitetura do Modelo:')
modelo.summary()
print()

# Calcular parâmetros manualmente para entender
params_embed  = VOCAB_SIZE * EMBED_DIM
params_densa1 = EMBED_DIM * UNITS_OCULTAS + UNITS_OCULTAS  # pesos + bias
params_saida  = UNITS_OCULTAS * 1 + 1
print(f'📐 Contagem de parâmetros:')
print(f'   Embedding   : {VOCAB_SIZE} × {EMBED_DIM} = {params_embed:,}')
print(f'   Dense oculta: {EMBED_DIM} × {UNITS_OCULTAS} + {UNITS_OCULTAS} = {params_densa1:,}')
print(f'   Dense saída : {UNITS_OCULTAS} × 1 + 1 = {params_saida:,}')
print(f'   TOTAL       : {params_embed + params_densa1 + params_saida:,} parâmetros treináveis')

---
### 3.2 Treinando o Modelo

In [ ]:
# ─────────────────────────────────────────────────────────────
# TREINAMENTO
#
# model.fit() executa o treinamento:
#   - Forward pass: calcula predições
#   - Calcula loss (diferença entre predição e rótulo real)
#   - Backward pass (backpropagation): ajusta os pesos
#   - Repete por N épocas
#
# epochs      → quantas vezes percorrer todo o dataset
# batch_size  → quantos exemplos processar por vez
# validation_data → monitora performance em dados não vistos
# callbacks   → ações automáticas durante o treino
# ─────────────────────────────────────────────────────────────

# Callback: parar o treino automaticamente se não melhorar
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',   # monitorar acurácia de validação
    patience=10,              # aguardar 10 épocas sem melhora
    restore_best_weights=True # restaurar o melhor modelo
)

print('🎓 Iniciando treinamento...')
print(f'   Épocas máximas  : 80')
print(f'   Batch size      : {len(X_treino)} (todo o treino de uma vez)')
print(f'   Early stopping  : patience=10')
print()

historico = modelo.fit(
    X_treino, y_treino,
    epochs=80,
    batch_size=len(X_treino),
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=0  # silencioso durante treino
)

n_epocas = len(historico.history['accuracy'])
acc_treino_final = historico.history['accuracy'][-1]
acc_val_final    = historico.history['val_accuracy'][-1]

print(f'✅ Treinamento concluído em {n_epocas} épocas!')
print(f'   Acurácia treino     : {acc_treino_final:.2%}')
print(f'   Acurácia validação  : {acc_val_final:.2%}')

---
### 3.3 Analisando as Curvas de Aprendizado

In [ ]:
# ─────────────────────────────────────────────────────────────
# CURVAS DE APRENDIZADO
# Visualiza a evolução da acurácia e loss ao longo das épocas
#
# Padrões a observar:
#   ✅ Treino e validação próximos → modelo generaliza bem
#   ⚠️  Treino sobe, validação desce → overfitting!
#   ❌  Ambas estagnadas → modelo não está aprendendo
# ─────────────────────────────────────────────────────────────

epocas = range(1, len(historico.history['accuracy']) + 1)

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(13, 9))

# ── Gráfico 1: Acurácia
ax1.plot(epocas, historico.history['accuracy'],     color='#1D9E75', lw=2, label='Treino')
ax1.plot(epocas, historico.history['val_accuracy'], color='#534AB7', lw=2, linestyle='--', label='Validação')
ax1.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_title('Acurácia por Época', fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('Acurácia')
ax1.set_ylim([0, 1.05])
ax1.legend()
ax1.grid(alpha=0.3)

# ── Gráfico 2: Loss
ax2.plot(epocas, historico.history['loss'],     color='#D85A30', lw=2, label='Treino')
ax2.plot(epocas, historico.history['val_loss'], color='#BA7517', lw=2, linestyle='--', label='Validação')
ax2.axhline(y=0.0, color='gray', linestyle=':', alpha=0.5)
ax2.set_title('Loss (Perda) por Época', fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(alpha=0.3)

# ── Gráfico 3: Diferença Treino vs Validação (gap de overfitting)
gap_acc  = np.array(historico.history['accuracy']) - np.array(historico.history['val_accuracy'])
cores_gap = ['#D85A30' if g > 0.15 else '#1D9E75' for g in gap_acc]
ax3.bar(epocas, gap_acc, color=cores_gap, alpha=0.8)
ax3.axhline(y=0, color='gray', lw=1)
ax3.axhline(y=0.15, color='#D85A30', lw=1, linestyle='--', label='Limiar overfitting')
ax3.set_title('Gap de Overfitting (treino - validação)', fontweight='bold')
ax3.set_xlabel('Época')
ax3.set_ylabel('Diferença de acurácia')
ax3.legend()
ax3.grid(alpha=0.3)

# ── Gráfico 4: Resumo final
metricas_nomes  = ['Acc\nTreino', 'Acc\nValidação', 'Loss\nTreino', 'Loss\nValidação']
metricas_vals   = [
    historico.history['accuracy'][-1],
    historico.history['val_accuracy'][-1],
    1 - historico.history['loss'][-1],
    1 - historico.history['val_loss'][-1],
]
cores_metricas = ['#1D9E75', '#534AB7', '#D85A30', '#BA7517']
bars = ax4.bar(metricas_nomes, metricas_vals, color=cores_metricas, alpha=0.85)
ax4.set_ylim([0, 1.15])
ax4.set_title('Resumo das Métricas Finais', fontweight='bold')
ax4.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, metricas_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.2%}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Análise Completa do Treinamento — Classificador de Sentimento',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
### 3.4 Fazendo Predições e Avaliando

In [ ]:
# ─────────────────────────────────────────────────────────────
# PREDIÇÕES E AVALIAÇÃO
#
# model.predict() → retorna probabilidade entre 0 e 1
#   > 0.5 → POSITIVO
#   < 0.5 → NEGATIVO
# ─────────────────────────────────────────────────────────────

# Frases de teste completamente novas
frases_teste = [
    'this is an absolutely amazing and wonderful film',
    'a terrible and boring movie i hated it',
    'the movie was decent with good acting',
    'outstanding performance and beautiful cinematography loved',
    'very disappointing and poorly written film awful',
    'i really enjoyed this brilliant and touching story',
    'complete waste of time horrible and very bad',
]

predicoes = modelo.predict(np.array(frases_teste), verbose=0)

print('🔮 Predições do Modelo:\n')
print(f'  {"Frase":<48} {"Prob.":<8} {"Classe":<14} {"Confiança"}')
print('  ' + '-' * 80)

for frase, pred in zip(frases_teste, predicoes):
    prob    = float(pred[0])
    classe  = 'POSITIVO 😊' if prob > 0.5 else 'NEGATIVO 😞'
    conf    = prob if prob > 0.5 else (1 - prob)
    barra   = '█' * int(conf * 20) + '░' * (20 - int(conf * 20))
    print(f'  {frase[:46]:<48} {prob:.4f}  {classe:<14} {barra}')

# Matriz de confusão simples na validação
print('\n📊 Avaliação na validação:')
preds_val = (modelo.predict(X_val, verbose=0) > 0.5).astype(int).flatten()

tp = sum((preds_val == 1) & (y_val == 1))  # Verdadeiro Positivo
tn = sum((preds_val == 0) & (y_val == 0))  # Verdadeiro Negativo
fp = sum((preds_val == 1) & (y_val == 0))  # Falso Positivo
fn = sum((preds_val == 0) & (y_val == 1))  # Falso Negativo

print(f'   Verdadeiro Positivo (TP): {tp}')
print(f'   Verdadeiro Negativo (TN): {tn}')
print(f'   Falso Positivo      (FP): {fp}')
print(f'   Falso Negativo      (FN): {fn}')
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f'\n   Precisão  : {precision:.2%}')
print(f'   Recall    : {recall:.2%}')
print(f'   F1-Score  : {f1:.2%}')

---
## 🔴 Parte 4 — Visualizando o que o Modelo Aprendeu

In [ ]:
# ─────────────────────────────────────────────────────────────
# EMBEDDINGS APRENDIDOS
# Após o treinamento, podemos inspecionar os vetores
# aprendidos pela camada Embedding
# Palavras de sentimentos similares devem ficar próximas!
# ─────────────────────────────────────────────────────────────

from sklearn.decomposition import PCA

# Extrair pesos da camada Embedding
pesos_embedding = modelo.get_layer('embedding').get_weights()[0]
vocab_modelo    = vetorizador.get_vocabulary()

print(f'📐 Matriz de Embeddings aprendida:')
print(f'   Shape: {pesos_embedding.shape}  ({VOCAB_SIZE} palavras × {EMBED_DIM} dimensões)')
print()

# Palavras de interesse para visualizar
palavras_positivas = ['wonderful', 'amazing', 'brilliant', 'excellent', 'loved', 'beautiful']
palavras_negativas = ['terrible', 'awful', 'boring', 'horrible', 'worst', 'disappointing']
palavras_neutras   = ['film', 'movie', 'story', 'acting', 'plot', 'characters']

# Encontrar índices no vocabulário
def obter_embedding_palavra(palavra):
    if palavra in vocab_modelo:
        idx = vocab_modelo.index(palavra)
        return pesos_embedding[idx]
    return None

# Coletar embeddings e rótulos
embeddings_viz, labels_viz, cores_viz = [], [], []
grupos_viz = [
    (palavras_positivas, '#1D9E75', '😊 Positivas'),
    (palavras_negativas, '#D85A30', '😞 Negativas'),
    (palavras_neutras,   '#534AB7', '😐 Neutras'),
]

for palavras, cor, nome in grupos_viz:
    for palavra in palavras:
        emb = obter_embedding_palavra(palavra)
        if emb is not None:
            embeddings_viz.append(emb)
            labels_viz.append(palavra)
            cores_viz.append(cor)

if len(embeddings_viz) >= 4:
    # Reduzir para 2D com PCA
    pca = PCA(n_components=2, random_state=42)
    emb_2d = pca.fit_transform(np.array(embeddings_viz))

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
               c=cores_viz, s=150, alpha=0.85, edgecolors='white', zorder=3)

    for i, (x, y) in enumerate(emb_2d):
        ax.annotate(labels_viz[i], (x, y),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=10, fontweight='bold')

    # Legenda
    patches = [mpatches.Patch(color=c, label=n) for _, c, n in grupos_viz]
    ax.legend(handles=patches, fontsize=10, loc='upper left')
    ax.set_title('Embeddings Aprendidos pelo Modelo (PCA 2D)\n'
                 'Palavras similares devem ficar agrupadas após o treino',
                 fontweight='bold', pad=12)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)')
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Poucas palavras encontradas no vocabulário para visualização.')
    print(f'   Vocabulário do modelo: {vocab_modelo[:20]}')

---
## 🟣 Parte 5 — Salvando e Carregando Modelos

In [ ]:
# ─────────────────────────────────────────────────────────────
# SALVAR E CARREGAR MODELOS
# Após treinar, salvamos o modelo para reutilizar sem retreinar
#
# Formatos disponíveis:
#   .keras  → formato nativo Keras (recomendado)
#   .h5     → formato HDF5 (legado, ainda suportado)
#   SavedModel → formato TensorFlow para deployment
# ─────────────────────────────────────────────────────────────

import os

# ── Salvar o modelo
caminho_modelo = 'modelo_sentimento.keras'
modelo.save(caminho_modelo)
tamanho_kb = os.path.getsize(caminho_modelo) / 1024
print(f'💾 Modelo salvo: "{caminho_modelo}" ({tamanho_kb:.1f} KB)')
print()

# ── Carregar o modelo
modelo_carregado = keras.models.load_model(caminho_modelo)
print(f'✅ Modelo carregado com sucesso!')
print(f'   Camadas: {len(modelo_carregado.layers)}')
print()

# ── Verificar que as predições são idênticas
frase_teste = np.array(['this is a wonderful and amazing film'])
pred_original  = float(modelo.predict(frase_teste, verbose=0)[0][0])
pred_carregado = float(modelo_carregado.predict(frase_teste, verbose=0)[0][0])

print(f'🔮 Verificação de consistência:')
print(f'   Frase          : "{frase_teste[0]}"')
print(f'   Modelo original: {pred_original:.6f}')
print(f'   Modelo carregado: {pred_carregado:.6f}')
print(f'   Idênticos      : {abs(pred_original - pred_carregado) < 1e-5}')
print()

# ── Salvar apenas os pesos (útil para transferir entre arquiteturas)
modelo.save_weights('pesos_sentimento.weights.h5')
print('💾 Pesos salvos separadamente: "pesos_sentimento.weights.h5"')
print()
print('💡 Boas práticas de salvamento:')
print('   .keras     → modelo completo (arquitetura + pesos + config)')
print('   .weights.h5 → apenas os pesos (precisa recriar a arquitetura)')
print('   SavedModel → formato para TF Serving em produção')

---
## 📖 Parte 6 — Comparativo: Abordagens de Vetorização

In [ ]:
# ─────────────────────────────────────────────────────────────
# COMPARATIVO: BoW vs TF-IDF vs Embedding
# Treina 3 modelos com diferentes representações
# e compara suas performances
# ─────────────────────────────────────────────────────────────

def criar_modelo_bow_tfidf(modo_vetorizacao, nome):
    """Cria um modelo com Bag-of-Words ou TF-IDF."""
    vet = keras.layers.TextVectorization(
        max_tokens=200,
        output_mode=modo_vetorizacao,
        standardize='lower_and_strip_punctuation',
    )
    vet.adapt(X_treino)

    m = keras.Sequential([
        vet,
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1,  activation='sigmoid'),
    ], name=nome)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

def criar_modelo_embedding(nome):
    """Cria um modelo com camada Embedding."""
    vet = keras.layers.TextVectorization(
        max_tokens=200, output_mode='int',
        output_sequence_length=15,
        standardize='lower_and_strip_punctuation',
    )
    vet.adapt(X_treino)
    m = keras.Sequential([
        vet,
        keras.layers.Embedding(200, 16),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dense(1,  activation='sigmoid'),
    ], name=nome)
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return m

configs = [
    ('Bag-of-Words', lambda: criar_modelo_bow_tfidf('count',  'bow'),    '#1D9E75'),
    ('TF-IDF',       lambda: criar_modelo_bow_tfidf('tf_idf', 'tfidf'),  '#534AB7'),
    ('Embedding',    lambda: criar_modelo_embedding('emb'),               '#D85A30'),
]

resultados_comparativo = {}
print('⚡ Treinando 3 modelos para comparação...')

historicos_comp = {}
for nome, criar_fn, _ in configs:
    m = criar_fn()
    h = m.fit(X_treino, y_treino, epochs=60, batch_size=20,
              validation_data=(X_val, y_val),
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy',
                                                        patience=8,
                                                        restore_best_weights=True)],
              verbose=0)
    acc_final = max(h.history['val_accuracy'])
    resultados_comparativo[nome] = acc_final
    historicos_comp[nome] = h
    print(f'   {nome:<16} → val_accuracy: {acc_final:.2%}')

# Visualizar comparativo
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: Barras de acurácia final
nomes_comp = list(resultados_comparativo.keys())
accs_comp  = list(resultados_comparativo.values())
cores_comp = [c for _, _, c in configs]
bars = ax1.bar(nomes_comp, accs_comp, color=cores_comp, alpha=0.85, edgecolor='white')
ax1.set_ylim([0, 1.15])
ax1.set_title('Acurácia de Validação\npor Estratégia de Vetorização', fontweight='bold')
ax1.set_ylabel('Acurácia de Validação')
ax1.grid(alpha=0.3, axis='y')
for bar, acc in zip(bars, accs_comp):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.1%}', ha='center', fontweight='bold', fontsize=12)

# Gráfico 2: Curvas de validação
for (nome, _, cor), hist in zip(configs, historicos_comp.values()):
    ax2.plot(hist.history['val_accuracy'], color=cor, lw=2, label=nome)
ax2.set_title('Evolução da Acurácia de Validação\npor Época', fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Acurácia de Validação')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.suptitle('Comparativo: Bag-of-Words vs TF-IDF vs Embedding',
             fontweight='bold', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 📝 Resumo Final

### Tabela de Referência Rápida

| Conceito | Keras | O que faz |
|---|---|---|
| Vetorizar texto | `TextVectorization(output_mode='count')` | Texto → vetor de contagens |
| TF-IDF | `TextVectorization(output_mode='tf_idf')` | Texto → pesos por raridade |
| Sequência | `TextVectorization(output_mode='int')` | Texto → IDs em ordem |
| Embedding | `Embedding(vocab, dim)` | IDs → vetores semânticos treináveis |
| Agregar sequência | `GlobalAveragePooling1D()` | Sequência → vetor único |
| Camada densa | `Dense(n, activation='relu')` | Transformação não-linear |
| Regularização | `Dropout(rate)` | Evita overfitting |
| Saída binária | `Dense(1, activation='sigmoid')` | Probabilidade entre 0 e 1 |
| Compilar | `model.compile(optimizer, loss, metrics)` | Configura o aprendizado |
| Treinar | `model.fit(X, y, epochs, callbacks)` | Ajusta os pesos |
| Salvar | `model.save('modelo.keras')` | Persiste modelo em disco |
| Carregar | `keras.models.load_model('modelo.keras')` | Restaura modelo salvo |

---

### Quando usar cada representação?

```
Bag-of-Words → baseline rápido, datasets pequenos, interpretabilidade importante
TF-IDF       → busca de informação, documentos longos, vocabulário especializado
Embedding    → quando a ORDER importa, datasets maiores, tarefas mais complexas
BERT/HF      → máxima performance, contexto bidirecional, modelos modernos (Aula 4)
```

---

## 🔜 Próximos Passos

Na **Aula 3** vamos adicionar memória às redes neurais com **RNNs, LSTMs e GRUs**,  
e entender o **mecanismo de atenção** que deu origem aos Transformers e ao BERT.